# Deepfake Forensics Analysis (Goal 7)

Uses CRISE saliency maps to diagnose *why* synthetic probes succeed or fail at fooling ArcFace.

**Pipeline:**
1. Match synthetic probe saliency maps to real probe saliency maps (same identity)
2. Compute saliency divergence: cosine similarity + L1
3. Establish similarity threshold empirically from the distribution
4. Classify every synthetic probe into case A / B / C / D
5. Per-region importance analysis (5 facial zones on 112×112 aligned chips)
6. Cross-method comparison table + 8+ figures

**Case taxonomy:**

| Case | Fooled ArcFace? | Saliency similar? | Interpretation |
|---|---|---|---|
| A | Yes | Yes | Fooled for right reasons — genuine identity features replicated |
| B | Yes | No  | **Fooled for wrong reasons** — exploiting artifacts / non-identity regions |
| C | No  | Yes | Correct features but insufficient identity transfer |
| D | No  | No  | Complete failure |

**Inputs required (must exist before running):**
- `data/synthetic_probes/metadata.csv`
- `results/crise_maps/synth_saliency_index.csv`
- `results/crise_maps/summary_crise_tau0.1_N1000_s8_p0.5_MASTERSEED123.csv`
- `results/crise_maps/*_saliency_norm.npy`

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

META_PATH        = "data/synthetic_probes/metadata.csv"
SAL_INDEX_PATH   = "results/crise_maps/synth_saliency_index.csv"
REAL_SUMMARY     = "results/crise_maps/summary_crise_tau0.1_N1000_s8_p0.5_MASTERSEED123.csv"
CRISE_MAP_DIR    = "results/crise_maps"

FIG_DIR          = "results/forensics_figures"

# Primary alpha/strength for per-method comparisons
PRIMARY_ALPHA    = 0.5

# Saliency similarity threshold — set empirically in cell below
# (placeholder; overwritten after inspecting the distribution)
SIM_THRESHOLD    = 0.75

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm import tqdm
from scipy import stats

os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# Facial region masks (fixed in 112×112 aligned chip space)
# Based on _DST_LANDMARKS positions from rise.py:
#   left eye (38,52), right eye (74,52), nose (56,72),
#   mouth-L (42,92), mouth-R (71,92)
# ---------------------------------------------------------------------------

def make_region_masks(h=112, w=112):
    """Return dict of (h,w) float32 binary region masks for the aligned chip."""
    masks = {}
    y, x = np.mgrid[0:h, 0:w]

    def ellipse(cy, cx, ry, rx):
        return (((y - cy) / ry) ** 2 + ((x - cx) / rx) ** 2 <= 1).astype(np.float32)

    # Eye zone: covers both eyes + brow area
    masks["eyes"]     = np.clip(ellipse(50, 38, 18, 20) + ellipse(50, 74, 18, 20), 0, 1)
    # Nose zone
    masks["nose"]     = ellipse(72, 56, 16, 14).astype(np.float32)
    # Mouth zone
    masks["mouth"]    = ellipse(92, 56, 14, 22).astype(np.float32)
    # Forehead / upper face (above eyes)
    masks["forehead"] = ((y < 38) & (x > 15) & (x < 97)).astype(np.float32)
    # Jaw / chin (below mouth)
    masks["jaw_chin"] = ((y > 98) & (x > 15) & (x < 97)).astype(np.float32)

    return masks


REGION_MASKS  = make_region_masks()
REGION_NAMES  = ["forehead", "eyes", "nose", "mouth", "jaw_chin"]

def region_importance(sal_norm):
    """Return dict of fractional saliency weight per region."""
    total = sal_norm.sum() + 1e-12
    return {r: float((sal_norm * REGION_MASKS[r]).sum() / total) for r in REGION_NAMES}


# Quick sanity: visualise masks
fig, axes = plt.subplots(1, len(REGION_NAMES), figsize=(12, 2.5))
for ax, name in zip(axes, REGION_NAMES):
    ax.imshow(REGION_MASKS[name], cmap="viridis", vmin=0, vmax=1)
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.suptitle("Region masks (112×112 aligned chip)", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_region_masks.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------

meta      = pd.read_csv(META_PATH)
sal_index = pd.read_csv(SAL_INDEX_PATH)
real_sum  = pd.read_csv(REAL_SUMMARY)

# Normalise column name for real summary
if "true_id" not in real_sum.columns:
    real_sum = real_sum.rename(columns={real_sum.columns[0]: "true_id"})

print(f"Synthetic probes      : {len(meta)}")
print(f"Saliency index entries: {len(sal_index)}")
print(f"Real probe CRISE rows : {len(real_sum)}")

In [ ]:
# ---------------------------------------------------------------------------
# Build per-identity real probe saliency map (mean over all real probes)
# ---------------------------------------------------------------------------

# real_sum must have a 'saliency_path' or we reconstruct from the file naming convention
# Try 'saliency_path' column first; fall back to scanning crise_maps dir

real_sal_by_id = {}   # identity -> mean saliency map (112×112 float32)

if "saliency_path" in real_sum.columns:
    sal_col = "saliency_path"
else:
    # Column may be named differently — find any column ending in .npy path
    npy_cols = [c for c in real_sum.columns if real_sum[c].astype(str).str.endswith(".npy").any()]
    sal_col = npy_cols[0] if npy_cols else None

if sal_col:
    for _, row in tqdm(real_sum.iterrows(), total=len(real_sum), desc="Loading real saliency maps"):
        tid  = row["true_id"]
        path = row[sal_col]
        if not isinstance(path, str) or not os.path.exists(path):
            continue
        sal = np.load(path).astype(np.float32)
        if tid not in real_sal_by_id:
            real_sal_by_id[tid] = []
        real_sal_by_id[tid].append(sal)

    # Mean over all real probes per identity
    real_sal_by_id = {tid: np.mean(maps, axis=0)
                      for tid, maps in real_sal_by_id.items()}
    print(f"Loaded real saliency maps for {len(real_sal_by_id)} identities")
else:
    print("[WARN] No saliency_path column found in real summary — skipping real map load.")
    print("       Update REAL_SUMMARY to point to a CSV that includes saliency_path.")

In [ ]:
# ---------------------------------------------------------------------------
# Compute saliency divergence for each synthetic probe
# ---------------------------------------------------------------------------

rows_out = []

for _, srow in tqdm(sal_index.iterrows(), total=len(sal_index), desc="Saliency divergence"):
    tid       = srow["identity"]
    sal_path  = srow["saliency_path"]

    if not isinstance(sal_path, str) or not os.path.exists(sal_path):
        continue

    synth_sal = np.load(sal_path).astype(np.float32).ravel()
    synth_sal_2d = synth_sal.reshape(112, 112)

    real_sal_2d  = real_sal_by_id.get(tid)
    if real_sal_2d is None:
        continue

    real_flat = real_sal_2d.ravel()

    # Cosine similarity (both already in [0,1], normalise for dot product)
    s_norm = synth_sal / (np.linalg.norm(synth_sal) + 1e-12)
    r_norm = real_flat / (np.linalg.norm(real_flat) + 1e-12)
    cos_sim = float(np.dot(s_norm, r_norm))
    l1_dist = float(np.abs(synth_sal - real_flat).mean())

    # Per-region importance
    reg_imp = region_importance(synth_sal_2d)

    rows_out.append(dict(
        output_path      = srow["output_path"],
        identity         = tid,
        generation_method= srow["generation_method"],
        blend_alpha      = srow["blend_alpha"],
        saliency_path    = sal_path,
        saliency_cosine_sim = cos_sim,
        saliency_l1         = l1_dist,
        **{f"reg_{k}": v for k, v in reg_imp.items()},
    ))

div_df = pd.DataFrame(rows_out)
print(f"Divergence computed for {len(div_df)} synthetic probes")

In [ ]:
# ---------------------------------------------------------------------------
# Merge divergence back into metadata
# ---------------------------------------------------------------------------

meta_upd = meta.merge(
    div_df[["output_path", "saliency_cosine_sim", "saliency_l1"] +
           [f"reg_{r}" for r in REGION_NAMES]],
    on="output_path", how="left",
)

# Overwrite the placeholder columns with real values
meta_upd["saliency_cosine_sim"] = meta_upd["saliency_cosine_sim_y"].combine_first(
    meta_upd["saliency_cosine_sim_x"]
)
meta_upd["saliency_l1"] = meta_upd["saliency_l1_y"].combine_first(
    meta_upd["saliency_l1_x"]
)
drop_cols = [c for c in meta_upd.columns if c.endswith("_x") or c.endswith("_y")]
meta_upd.drop(columns=drop_cols, inplace=True)

meta_upd.to_csv(META_PATH, index=False)
print(f"metadata.csv updated with saliency divergence → {META_PATH}")

In [ ]:
# ---------------------------------------------------------------------------
# Fig 1: Saliency cosine similarity distribution
# Used to set SIM_THRESHOLD empirically
# ---------------------------------------------------------------------------

sims = div_df["saliency_cosine_sim"].dropna()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(sims, bins=40, color="steelblue", edgecolor="white", linewidth=0.4)
ax.axvline(sims.median(), color="red",    ls="--", lw=1.5, label=f"Median = {sims.median():.3f}")
ax.axvline(sims.quantile(0.25), color="orange", ls=":", lw=1.2,
           label=f"25th pct = {sims.quantile(0.25):.3f}")
ax.set_xlabel("Cosine similarity (synthetic vs real probe saliency maps)")
ax.set_ylabel("Count")
ax.set_title("Saliency map similarity distribution — all synthetic probes")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig1_saliency_sim_dist.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"mean={sims.mean():.3f}  median={sims.median():.3f}  std={sims.std():.3f}")
print(f"Suggested threshold (median): {sims.median():.3f}")
print(">>> Update SIM_THRESHOLD in Config cell if needed, then re-run from the case classification cell.")

In [ ]:
# ---------------------------------------------------------------------------
# Case classification  (update SIM_THRESHOLD above if needed)
# ---------------------------------------------------------------------------

analysis = meta_upd[
    meta_upd["embedding_ok"].astype(bool) &
    meta_upd["rank1_match"].notna() &
    meta_upd["saliency_cosine_sim"].notna()
].copy()

analysis["sal_similar"] = analysis["saliency_cosine_sim"] >= SIM_THRESHOLD

def classify_case(row):
    if row["rank1_match"] and row["sal_similar"]:     return "A"
    if row["rank1_match"] and not row["sal_similar"]: return "B"
    if not row["rank1_match"] and row["sal_similar"]: return "C"
    return "D"

analysis["case_label"] = analysis.apply(classify_case, axis=1)

# Write case labels back to metadata
meta_upd.loc[analysis.index, "case_label"] = analysis["case_label"]
meta_upd.to_csv(META_PATH, index=False)

print(f"Classified {len(analysis)} probes  (threshold={SIM_THRESHOLD})")
print(analysis["case_label"].value_counts().sort_index().to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Fig 2: Cross-method comparison table (Case A/B/C/D % + Rank-1 rate)
# Primary alpha/strength only
# ---------------------------------------------------------------------------

primary = analysis[
    (analysis["blend_alpha"] == PRIMARY_ALPHA) |
    (analysis["generation_method"] == "insightface_swap")  # no alpha for swap
].copy()

rows_table = []
for method, grp in primary.groupby("generation_method"):
    total = len(grp)
    row = {"Method": method, "n": total,
           "Rank-1 rate": grp["rank1_match"].mean()}
    for case in ["A", "B", "C", "D"]:
        row[f"Case {case} %"] = (grp["case_label"] == case).sum() / total * 100
    rows_table.append(row)

table_df = pd.DataFrame(rows_table).set_index("Method")
print("=== Cross-method comparison ===")
print(table_df.round(3).to_string())

# Bar chart
case_cols = ["Case A %", "Case B %", "Case C %", "Case D %"]
colors    = ["#2ecc71", "#e74c3c", "#f39c12", "#95a5a6"]

fig, ax = plt.subplots(figsize=(8, 4))
table_df[case_cols].plot(kind="bar", ax=ax, color=colors, edgecolor="white", width=0.7)
ax.set_ylabel("% of probes")
ax.set_title(f"Case distribution per generation method (α/strength = {PRIMARY_ALPHA})")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig2_cross_method_cases.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Fig 3: Per-region importance profiles by case
# ---------------------------------------------------------------------------

reg_cols = [f"reg_{r}" for r in REGION_NAMES]
case_reg = analysis.groupby("case_label")[reg_cols].mean()

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(REGION_NAMES))
width = 0.18
case_colors = {"A": "#2ecc71", "B": "#e74c3c", "C": "#f39c12", "D": "#95a5a6"}

for i, (case, row_) in enumerate(case_reg.iterrows()):
    ax.bar(x + i * width, row_.values, width, label=f"Case {case}",
           color=case_colors.get(case, "gray"), edgecolor="white")

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(REGION_NAMES)
ax.set_ylabel("Mean fractional saliency weight")
ax.set_title("Per-region importance by forensic case")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig3_region_importance_by_case.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\nMean per-region importance by case:")
print(case_reg.round(4).to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Fig 4: Saliency similarity vs ArcFace similarity (scatter, coloured by case)
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6, 5))
for case, grp in analysis.groupby("case_label"):
    ax.scatter(grp["arcface_similarity"], grp["saliency_cosine_sim"],
               label=f"Case {case}", alpha=0.5, s=18,
               color=case_colors.get(case, "gray"))

ax.axhline(SIM_THRESHOLD, color="black", ls="--", lw=1, label=f"Threshold={SIM_THRESHOLD:.2f}")
ax.set_xlabel("ArcFace cosine similarity to true identity")
ax.set_ylabel("Saliency map cosine similarity (synthetic vs real)")
ax.set_title("ArcFace confidence vs saliency faithfulness")
ax.legend(fontsize=8, markerscale=1.5)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig4_sim_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Fig 5 & 6: Alpha/strength ablation — rank-1 rate and case distribution
# ---------------------------------------------------------------------------

for method in analysis["generation_method"].unique():
    grp_m = analysis[analysis["generation_method"] == method]
    if grp_m["blend_alpha"].nunique() < 2:
        continue

    ablation_rows = []
    for a, grp_a in grp_m.groupby("blend_alpha"):
        r = {"alpha": a, "rank1_rate": grp_a["rank1_match"].mean()}
        for case in ["A", "B", "C", "D"]:
            r[f"case_{case}"] = (grp_a["case_label"] == case).mean() * 100
        ablation_rows.append(r)
    abl_df = pd.DataFrame(ablation_rows).sort_values("alpha")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))

    ax1.plot(abl_df["alpha"], abl_df["rank1_rate"], "o-", color="steelblue")
    ax1.set_xlabel("alpha / strength")
    ax1.set_ylabel("Rank-1 rate")
    ax1.set_title(f"{method} — fooling rate vs alpha")
    ax1.grid(alpha=0.3)

    for case, color in case_colors.items():
        ax2.plot(abl_df["alpha"], abl_df[f"case_{case}"], "o-",
                 color=color, label=f"Case {case}")
    ax2.set_xlabel("alpha / strength")
    ax2.set_ylabel("% of probes")
    ax2.set_title(f"{method} — case distribution vs alpha")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)

    plt.suptitle(f"Ablation: {method}", fontsize=11)
    plt.tight_layout()
    fname = f"fig_ablation_{method}.png"
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Figs 7+: Example saliency map pairs — 2 examples per case (8 figures)
# Shows: real probe saliency | synthetic saliency | overlay difference
# ---------------------------------------------------------------------------

N_EXAMPLES = 2

for case in ["A", "B", "C", "D"]:
    case_rows = analysis[analysis["case_label"] == case].dropna(
        subset=["saliency_path"]
    ).head(N_EXAMPLES)

    if len(case_rows) == 0:
        print(f"No examples for Case {case}")
        continue

    fig, axes = plt.subplots(len(case_rows), 3,
                             figsize=(9, 3 * len(case_rows)))
    if len(case_rows) == 1:
        axes = axes[np.newaxis, :]

    for row_i, (_, row_) in enumerate(case_rows.iterrows()):
        tid = row_["identity"]

        synth_sal = np.load(row_["saliency_path"]).astype(np.float32)
        real_sal  = real_sal_by_id.get(tid)

        diff = np.abs(synth_sal - real_sal) if real_sal is not None else np.zeros_like(synth_sal)

        titles = [
            f"Real probe\n{tid[:22]}",
            f"Synthetic ({row_['generation_method']})\ncos_sim={row_['saliency_cosine_sim']:.3f}",
            f"|Δ saliency|\nrank1={row_['rank1_match']}  case={case}",
        ]
        imgs = [real_sal if real_sal is not None else np.zeros_like(synth_sal),
                synth_sal, diff]

        for col_i, (img, title) in enumerate(zip(imgs, titles)):
            axes[row_i, col_i].imshow(img, cmap="hot", vmin=0, vmax=1)
            axes[row_i, col_i].set_title(title, fontsize=8)
            axes[row_i, col_i].axis("off")

    plt.suptitle(f"Case {case} examples", fontsize=11)
    plt.tight_layout()
    fname = f"fig_examples_case{case}.png"
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Case {case}: saved {fname}")

In [ ]:
# ---------------------------------------------------------------------------
# Summary table (paper-ready)
# ---------------------------------------------------------------------------

print("=== Full cross-method summary (all alphas/strengths) ===")
full_table = []
for method, grp in analysis.groupby("generation_method"):
    total = len(grp)
    r = {"Method": method, "n": total,
         "Rank-1 rate": f"{grp['rank1_match'].mean():.3f}"}
    for case in ["A", "B", "C", "D"]:
        pct = (grp["case_label"] == case).sum() / total * 100
        r[f"Case {case} %"] = f"{pct:.1f}"
    full_table.append(r)

print(pd.DataFrame(full_table).set_index("Method").to_string())

print("\n=== Primary alpha only ===")
print(table_df.round(3).to_string())

print(f"\nAll figures saved to: {FIG_DIR}/")